In [22]:
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import TIME_WINDOWS
from config.paths import ERPAC_DIR

Data extraction for LMMs -> **R analysis**

In [24]:
erpac_df = pd.read_parquet(
    r"F:\# study 2\eeg_data\erpac\erpac_results.parquet"
)

erpac_df

,sub,group,task,task_stage,coupling,roi,amp_freq,time,erpac_value
0,s1_pac_sub01,Y,FTT,plan,theta_gamma,M1,32.5,-0.500,0.122996
1,s1_pac_sub01,Y,FTT,plan,theta_gamma,M1,32.5,-0.498,0.116964
2,s1_pac_sub01,Y,FTT,plan,theta_gamma,M1,32.5,-0.496,0.120774
3,s1_pac_sub01,Y,FTT,plan,theta_gamma,M1,32.5,-0.494,0.126240
4,s1_pac_sub01,Y,FTT,plan,theta_gamma,M1,32.5,-0.492,0.130517
...,...,...,...,...,...,...,...,...,...
25430755,s1_pac_sub68,O,FTT,go,beta_gamma,SMA,76.5,0.492,0.113945
25430756,s1_pac_sub68,O,FTT,go,beta_gamma,SMA,76.5,0.494,0.109723
25430757,s1_pac_sub68,O,FTT,go,beta_gamma,SMA,76.5,0.496,0.105073
25430758,s1_pac_sub68,O,FTT,go,beta_gamma,SMA,76.5,0.498,0.104456


AVERAGE ACROSS FREQS AND TIME WINDOWS

In [ ]:
def average_erpac_windows(erpac_df, time_windows=TIME_WINDOWS):

    df = erpac_df.copy()

    def assign_time_window(row):

        stage = row["task_stage"]
        t = row["time"]

        for window_name, (tmin, tmax) in time_windows[stage].items():
            if tmin <= t < tmax:
                return window_name

        return np.nan

    df["time_win"] = df.apply(
        assign_time_window,
        axis=1
    )

    df = df.dropna(
        subset=["time_win"]
    )

    return (
        df
        .groupby(
            [
                "sub",
                "group",
                "task",
                "task_stage",
                "coupling",
                "roi",
                "time_win",
            ],
            as_index=False,
        )
        ["erpac_value"]
        .mean()
    )


erpac_window_df = average_erpac_windows(erpac_df)
erpac_window_df

,sub,group,task,task_stage,coupling,roi,time_win,erpac_value
0,s1_pac_sub01,Y,FTT,go,alpha_gamma,M1,early_post,0.133307
1,s1_pac_sub01,Y,FTT,go,alpha_gamma,M1,late_post,0.140503
2,s1_pac_sub01,Y,FTT,go,alpha_gamma,M1,move,0.143815
3,s1_pac_sub01,Y,FTT,go,alpha_gamma,M1,pre,0.140660
4,s1_pac_sub01,Y,FTT,go,alpha_gamma,PMC,early_post,0.135740
...,...,...,...,...,...,...,...,...
3943,s1_pac_sub77,Y,FTT,plan,theta_gamma,S1,late,0.123103
3944,s1_pac_sub77,Y,FTT,plan,theta_gamma,S1,middle,0.112072
3945,s1_pac_sub77,Y,FTT,plan,theta_gamma,SMA,early,0.113935
3946,s1_pac_sub77,Y,FTT,plan,theta_gamma,SMA,late,0.124789


In [ ]:
output_path = os.path.join(
    ERPAC_DIR,
    "erpac_window_averaged.csv"
)

erpac_window_df.to_csv(
    output_path,
    index=False
)

print(erpac_window_df.head())
print(erpac_window_df.shape)


            sub group task task_stage     coupling  roi    time_win  \
0  s1_pac_sub01     Y  FTT         go  alpha_gamma   M1  early_post   
1  s1_pac_sub01     Y  FTT         go  alpha_gamma   M1   late_post   
2  s1_pac_sub01     Y  FTT         go  alpha_gamma   M1        move   
3  s1_pac_sub01     Y  FTT         go  alpha_gamma   M1         pre   
4  s1_pac_sub01     Y  FTT         go  alpha_gamma  PMC  early_post   

   erpac_value  
0     0.133307  
1     0.140503  
2     0.143815  
3     0.140660  
4     0.135740  
(3948, 8)
Saved to: F:\# study 2\eeg_data\erpac\erpac_window_averaged.csv


AVERAGE ACROSS TIME

In [25]:
def average_erpac_stage(erpac_df):

    df = erpac_df.copy()

    # Keep only the desired time range for each stage
    mask = (
        (
            (df["task_stage"] == "plan") &
            (df["time"] >= 0.0) &
            (df["time"] <= 0.5)
        )
        |
        (
            (df["task_stage"] == "go") &
            (df["time"] >= -0.15) &
            (df["time"] <= 0.5)
        )
    )

    df = df[mask]

    # Average across both time and amplitude-frequency bins
    df_mean = (
        df
        .groupby(
            [
                "sub",
                "group",
                "task",
                "task_stage",
                "coupling",
                "roi",
            ],
            as_index=False,
        )["erpac_value"]
        .mean()
    )

    return df_mean

In [27]:
erpac_stage_mean_df = average_erpac_stage(erpac_df)

print(erpac_stage_mean_df.head())
print(erpac_stage_mean_df.shape)

output_path = os.path.join(
    ERPAC_DIR,
    "erpac_no_time.csv"
)

erpac_stage_mean_df.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

            sub group task task_stage     coupling  roi  erpac_value
0  s1_pac_sub01     Y  FTT         go  alpha_gamma   M1     0.139323
1  s1_pac_sub01     Y  FTT         go  alpha_gamma  PMC     0.138749
2  s1_pac_sub01     Y  FTT         go  alpha_gamma   S1     0.138485
3  s1_pac_sub01     Y  FTT         go  alpha_gamma  SMA     0.137439
4  s1_pac_sub01     Y  FTT         go   beta_gamma   M1     0.138114
(1128, 7)
Saved to: F:\# study 2\eeg_data\erpac\erpac_no_time.csv


WITH TIME WINDOWS AVERAGED ACROSS ROI

Prep done. Move to R for LMMs analysis